#**Logistic Regression**

In [ ]:
# ===============================
# Meter Remark Classification AI
# Logistic Regression + TF-IDF
# ===============================

# 1 Import Libraries
import pandas as pd
import pickle

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report


# 2 Load Dataset
data = pd.read_csv("dataset.csv")

print("Dataset Loaded Successfully")
print(data.head())


# 3 Define Input and Output
X = data['TEXT_REMARKS']
y = data['ACTION_CATEGORY']


# 4 TF-IDF Feature Extraction
tfidf = TfidfVectorizer()

X_tfidf = tfidf.fit_transform(X)

print("TF-IDF Vectorization Completed")


# 5 Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y,
    test_size=0.2,
    random_state=42
)

print("Dataset Split Completed")


# 6 Train Logistic Regression Model
model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

print("Model Training Completed")


# 7 Model Evaluation
y_pred = model.predict(X_test)

print("\nModel Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


# 8 Save Model and TF-IDF
pickle.dump(model, open("meter_remark_model.pkl", "wb"))
pickle.dump(tfidf, open("tfidf_vectorizer.pkl", "wb"))

print("\nModel Saved Successfully")


# 9 Prediction Function
def predict_department(text):

    text_vec = tfidf.transform([text])

    prediction = model.predict(text_vec)

    return prediction[0]


# 10 User Input System
print("\n===== Meter Remark Classification System =====")

while True:

    remark = input("\nEnter Meter Testing Remark (or type exit): ")

    if remark.lower() == "exit":
        break

    result = predict_department(remark)

    print("Concerned Department:", result)

Dataset Loaded Successfully
                                        TEXT_REMARKS  \
0  Consumer want Some time for meter testing as h...   
1  Consumer going to Village Consumer want some t...   
2  as per telephonic talk with consumer consumer ...   
3  No person at side and no response on said phon...   
4  No person at side and no response on said phon...   

                           ACTION_CATEGORY  
0  Back-Office / Administrative Operations  
1  Back-Office / Administrative Operations  
2  Back-Office / Administrative Operations  
3  Back-Office / Administrative Operations  
4  Back-Office / Administrative Operations  
TF-IDF Vectorization Completed
Dataset Split Completed
Model Training Completed

Model Accuracy: 0.9890939597315436

Classification Report:
                                         precision    recall  f1-score   support

Back-Office / Administrative Operations       0.94      0.96      0.95        46
     CEG / Enforcement / Vigilance Team       1.00      0.97  

In [ ]:
# Total Count of Dataset: 15524
# Tested OK: 12736
# Return Remarks Case: 2788

# Count of Remarks: 5959 (Tested OK Reduced Randomly Manualy) (75% Tested Ok Remarks has been dropped)
# 4767 for Testing (80%)
# 1192 for Training (20%)



#**SVM**

In [7]:
# =========================================================
# METER REMARK CLASSIFICATION SYSTEM (SVM + CSV UPLOAD)
# =========================================================

# FEATURES:
# 1. Train SVM Model
# 2. Save Model + TFIDF
# 3. Live Prediction
# 4. Upload CSV File
# 5. Auto Predict Department
# 6. Generate Output CSV
# =========================================================


# =========================
# 1 IMPORT LIBRARIES
# =========================

import pandas as pd
import pickle

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# =========================
# 2 LOAD DATASET
# =========================

data = pd.read_csv("dataset.csv")

print("Dataset Loaded Successfully")
print(data.head())


# =========================
# 3 INPUT AND OUTPUT
# =========================

X = data['TEXT_REMARKS']
y = data['ACTION_CATEGORY']


# =========================
# 4 TF-IDF VECTORIZATION
# =========================

tfidf = TfidfVectorizer(
    stop_words='english',
    lowercase=True
)

X_tfidf = tfidf.fit_transform(X)

print("\nTF-IDF Vectorization Completed")
print("Total Features:", X_tfidf.shape[1])


# =========================
# 5 TRAIN TEST SPLIT
# =========================

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y,
    test_size=0.2,
    random_state=42
)

print("\nDataset Split Completed")


# =========================
# 6 TRAIN SVM MODEL
# =========================

svm_model = SVC(
    kernel='linear',
    probability=True
)

svm_model.fit(X_train, y_train)

print("\nSVM Model Training Completed")


# =========================
# 7 MODEL PREDICTION
# =========================

y_pred = svm_model.predict(X_test)


# =========================
# 8 ACCURACY CHECK
# =========================

accuracy = accuracy_score(y_test, y_pred)

print("\n==============================")
print("SVM MODEL ACCURACY")
print("==============================")

print("Accuracy:", accuracy)


# =========================
# 9 CLASSIFICATION REPORT
# =========================

print("\nClassification Report:\n")

print(classification_report(y_test, y_pred))


# =========================
# 10 CONFUSION MATRIX
# =========================

print("\nConfusion Matrix:\n")

print(confusion_matrix(y_test, y_pred))


# =========================
# 11 SAVE MODEL
# =========================

pickle.dump(svm_model, open("svm_meter_model.pkl", "wb"))

pickle.dump(tfidf, open("svm_tfidf.pkl", "wb"))

print("\nSVM Model Saved Successfully")


# =========================================================
# 12 LIVE PREDICTION FUNCTION
# =========================================================

def predict_department(text):

    # Convert text into TF-IDF vector
    text_vector = tfidf.transform([str(text)])

    # Predict Department
    prediction = svm_model.predict(text_vector)

    # Prediction Probability
    probability = svm_model.predict_proba(text_vector)

    confidence = probability.max() * 100

    return prediction[0], confidence


# =========================================================
# 13 SINGLE LIVE PREDICTION
# =========================================================

print("\n====================================")
print("Meter Remark AI Classification System")
print("Type 'exit' to stop")
print("====================================")

while True:

    remark = input("\nEnter Meter Testing Remark: ")

    if remark.lower() == "exit":
        break

    department, confidence = predict_department(remark)

    print("\nPredicted Department:", department)

    print("Confidence Score:", round(confidence, 2), "%")


# =========================================================
# 14 BULK CSV FILE PREDICTION WITH EVALUATION
# =========================================================

print("\n====================================")
print("BULK CSV FILE PREDICTION SYSTEM")
print("====================================")

input_file = input("\nEnter CSV File Name: ")

# LOAD CSV
bulk_data = pd.read_csv(input_file)

print("\nCSV Loaded Successfully")
print(bulk_data.head())


# =========================================================
# CHECK REQUIRED COLUMN
# =========================================================

if 'TEXT_REMARKS' not in bulk_data.columns:

    print("\nERROR:")
    print("CSV must contain TEXT_REMARKS column")

else:

    predicted_departments = []
    confidence_scores = []
    confidence_ranges = []
    match_status = []

    # =====================================================
    # PREDICTION LOOP
    # =====================================================

    for index, row in bulk_data.iterrows():

        remark = row['TEXT_REMARKS']

        # Predict
        department, confidence = predict_department(remark)

        # Save Prediction
        predicted_departments.append(department)

        confidence = round(confidence, 2)

        confidence_scores.append(round(confidence, 2))

        # =================================================
        # CONFIDENCE RANGE CLASSIFICATION
        # =================================================

        if confidence >= 90:

            confidence_ranges.append("100-90% Very High")

        elif confidence >= 75:

            confidence_ranges.append("90-75% High")

        elif confidence >= 50:

            confidence_ranges.append("75-50% Medium")

        else:

            confidence_ranges.append("Below 50% Low")

        # =================================================
        # MATCH STATUS
        # =================================================

        if 'ACTION_CATEGORY' in bulk_data.columns:

            actual = row['ACTION_CATEGORY']

            if str(actual).strip() == str(department).strip():

                match_status.append("Correct")

            else:

                match_status.append("Wrong")

        else:

            match_status.append("No Actual Label")

    # =====================================================
    # ADD OUTPUT COLUMNS
    # =====================================================

    bulk_data['PREDICTED_DEPARTMENT'] = predicted_departments

    bulk_data['CONFIDENCE_SCORE'] = confidence_scores

    bulk_data['CONFIDENCE_RANGE'] = confidence_ranges

    bulk_data['MATCH_STATUS'] = match_status

    # =====================================================
    # SAVE OUTPUT CSV
    # =====================================================

    output_file = "Final_Predicted_Output.csv"

    bulk_data.to_csv(output_file, index=False)

    print("\n====================================")
    print("PREDICTION COMPLETED SUCCESSFULLY")
    print("====================================")

    print("\nOutput File Saved As:")
    print(output_file)

    # =====================================================
    # SUMMARY
    # =====================================================

    if 'ACTUAL_DEPARTMENT' in bulk_data.columns:

        correct_count = (bulk_data['MATCH_STATUS'] == "Correct").sum()

        wrong_count = (bulk_data['MATCH_STATUS'] == "Wrong").sum()

        print("\n====================================")
        print("PREDICTION SUMMARY")
        print("====================================")

        print("Correct Predictions :", correct_count)

        print("Wrong Predictions   :", wrong_count)

    print("\nSample Output:\n")

    print(bulk_data.head())
# =========================================================
# END OF PROJECT
# =========================================================

Dataset Loaded Successfully
                                        TEXT_REMARKS  \
0  Consumer want Some time for meter testing as h...   
1  Consumer going to Village Consumer want some t...   
2  as per telephonic talk with consumer consumer ...   
3  No person at side and no response on said phon...   
4  No person at side and no response on said phon...   

                           ACTION_CATEGORY  Unnamed: 2  Unnamed: 3  \
0  Back-Office / Administrative Operations         NaN         NaN   
1  Back-Office / Administrative Operations         NaN         NaN   
2  Back-Office / Administrative Operations         NaN         NaN   
3  Back-Office / Administrative Operations         NaN         NaN   
4  Back-Office / Administrative Operations         NaN         NaN   

   Unnamed: 4  Unnamed: 5  Unnamed: 6  Unnamed: 7  Unnamed: 8  Unnamed: 9  \
0         NaN         NaN         NaN         NaN         NaN         NaN   
1         NaN         NaN         NaN         NaN         Na

#**ANN**

In [2]:
# =========================================================
# METER REMARK CLASSIFICATION USING ANN
# =========================================================

# =========================
# 1 IMPORT LIBRARIES
# =========================

import pandas as pd
import pickle
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical


# =========================
# 2 LOAD DATASET
# =========================

data = pd.read_csv("dataset.csv")

print("Dataset Loaded Successfully")

print(data.head())


# =========================
# 3 INPUT AND OUTPUT
# =========================

X = data['TEXT_REMARKS']

y = data['ACTION_CATEGORY']


# =========================
# 4 TF-IDF VECTORIZATION
# =========================

tfidf = TfidfVectorizer(
    stop_words='english',
    lowercase=True
)

X_tfidf = tfidf.fit_transform(X)

print("\nTF-IDF Completed")

print("Total Features:", X_tfidf.shape[1])


# =========================
# 5 LABEL ENCODING
# =========================

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

# Convert into categorical
y_categorical = to_categorical(y_encoded)

print("\nLabel Encoding Completed")


# =========================
# 6 TRAIN TEST SPLIT
# =========================

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y_categorical,
    test_size=0.2,
    random_state=42
)

print("\nDataset Split Completed")


# =========================
# 7 BUILD ANN MODEL
# =========================

model = Sequential()

# Input Layer
model.add(Dense(
    128,
    activation='relu',
    input_shape=(X_train.shape[1],)
))

# Hidden Layer
model.add(Dense(
    64,
    activation='relu'
))

# Output Layer
model.add(Dense(
    y_categorical.shape[1],
    activation='softmax'
))

print("\nANN Model Created")


# =========================
# 8 COMPILE MODEL
# =========================

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("\nModel Compilation Completed")


# =========================
# 9 TRAIN MODEL
# =========================

history = model.fit(
    X_train.toarray(),
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2
)

print("\nANN Training Completed")


# =========================
# 10 MODEL PREDICTION
# =========================

y_pred_probs = model.predict(X_test.toarray())

y_pred = np.argmax(y_pred_probs, axis=1)

y_test_actual = np.argmax(y_test, axis=1)


# =========================
# 11 ACCURACY
# =========================

accuracy = accuracy_score(
    y_test_actual,
    y_pred
)

print("\n==============================")
print("ANN MODEL ACCURACY")
print("==============================")

print("Accuracy:", accuracy)


# =========================
# 12 CLASSIFICATION REPORT
# =========================

print("\nClassification Report:\n")

print(
    classification_report(
        y_test_actual,
        y_pred
    )
)


# =========================
# 13 CONFUSION MATRIX
# =========================

print("\nConfusion Matrix:\n")

print(
    confusion_matrix(
        y_test_actual,
        y_pred
    )
)


# =========================
# 14 SAVE MODEL
# =========================

model.save("ann_meter_model.h5")

pickle.dump(tfidf, open("ann_tfidf.pkl", "wb"))

pickle.dump(label_encoder, open("label_encoder.pkl", "wb"))

print("\nANN Model Saved Successfully")


# =========================================================
# 15 LIVE PREDICTION FUNCTION
# =========================================================

def predict_department(text):

    # TF-IDF Transform
    text_vector = tfidf.transform([str(text)])

    # Predict Probability
    prediction_prob = model.predict(
        text_vector.toarray()
    )

    # Get Highest Probability Index
    predicted_index = np.argmax(prediction_prob)

    # Decode Department Name
    predicted_department = label_encoder.inverse_transform(
        [predicted_index]
    )[0]

    # Confidence Score
    confidence = np.max(prediction_prob) * 100

    return predicted_department, confidence


# =========================================================
# 16 BULK CSV PREDICTION
# =========================================================

input_file = input("\nEnter CSV File Name: ")

bulk_data = pd.read_csv(input_file)

print("Total Rows in Uploaded CSV:", len(bulk_data))

predicted_departments = []

confidence_scores = []

confidence_ranges = []

match_status = []


# =========================================================
# PREDICTION LOOP
# =========================================================

for index, row in bulk_data.iterrows():

    remark = row['TEXT_REMARKS']

    department, confidence = predict_department(remark)

    confidence = round(confidence, 2)

    predicted_departments.append(department)

    confidence_scores.append(confidence)

    # =========================================
    # CONFIDENCE RANGE
    # =========================================

    if confidence >= 90:

        confidence_ranges.append("100-90% Very High")

    elif confidence >= 75:

        confidence_ranges.append("90-75% High")

    elif confidence >= 50:

        confidence_ranges.append("75-50% Medium")

    else:

        confidence_ranges.append("Below 50% Low")

    # =========================================
    # MATCH STATUS
    # =========================================

    actual = str(row['ACTION_CATEGORY']).strip()

    predicted = str(department).strip()

    if actual == predicted:

        match_status.append("Correct")

    else:

        match_status.append("Wrong")


# =========================================================
# ADD OUTPUT COLUMNS
# =========================================================

bulk_data['PREDICTED_DEPARTMENT'] = predicted_departments

bulk_data['CONFIDENCE_SCORE'] = confidence_scores

bulk_data['CONFIDENCE_RANGE'] = confidence_ranges

bulk_data['MATCH_STATUS'] = match_status

bulk_data['IS_MATCH'] = (
    bulk_data['ACTION_CATEGORY']
    ==
    bulk_data['PREDICTED_DEPARTMENT']
)


# =========================================================
# SAVE OUTPUT CSV
# =========================================================

bulk_data.to_csv(
    "ANN_Predicted_Output.csv",
    index=False
)

print("\n====================================")
print("ANN PREDICTION COMPLETED")
print("====================================")

print("\nOutput File Saved Successfully")

Dataset Loaded Successfully
                                        TEXT_REMARKS  \
0  Consumer want Some time for meter testing as h...   
1  Consumer going to Village Consumer want some t...   
2  as per telephonic talk with consumer consumer ...   
3  No person at side and no response on said phon...   
4  No person at side and no response on said phon...   

                           ACTION_CATEGORY  Unnamed: 2  Unnamed: 3  \
0  Back-Office / Administrative Operations         NaN         NaN   
1  Back-Office / Administrative Operations         NaN         NaN   
2  Back-Office / Administrative Operations         NaN         NaN   
3  Back-Office / Administrative Operations         NaN         NaN   
4  Back-Office / Administrative Operations         NaN         NaN   

   Unnamed: 4  Unnamed: 5  Unnamed: 6  Unnamed: 7  Unnamed: 8  Unnamed: 9  \
0         NaN         NaN         NaN         NaN         NaN         NaN   
1         NaN         NaN         NaN         NaN         Na

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



ANN Model Created

Model Compilation Completed
Epoch 1/10
120/120 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - accuracy: 0.8339 - loss: 0.7043 - val_accuracy: 0.9801 - val_loss: 0.1432
Epoch 2/10
120/120 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9801 - loss: 0.0932 - val_accuracy: 0.9916 - val_loss: 0.0514
Epoch 3/10
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9942 - loss: 0.0413 - val_accuracy: 0.9895 - val_loss: 0.0400
Epoch 4/10
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9961 - loss: 0.0255 - val_accuracy: 0.9906 - val_loss: 0.0361
Epoch 5/10
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9963 - loss: 0.0206 - val_accuracy: 0.9895 - val_loss: 0.0321
Epoch 6/10
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9971 - loss: 0.0172 - val_accuracy: 0.9895 - val_loss: 0.0329
Epoch 7/10
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9974 - loss: 0.0156 - val_accuracy: 0.9916 - val_loss: 0.0320
Epoch 8/10
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - a


ANN MODEL ACCURACY
Accuracy: 0.9907718120805369

Classification Report:

              precision    recall  f1-score   support

           0       0.94      1.00      0.97        46
           1       1.00      1.00      1.00        35
           2       1.00      1.00      1.00        32
           3       1.00      0.94      0.97        89
           4       0.99      0.99      0.99       337
           5       1.00      0.89      0.94         9
           6       0.99      1.00      1.00       644

    accuracy                           0.99      1192
   macro avg       0.99      0.97      0.98      1192
weighted avg       0.99      0.99      0.99      1192


Confusion Matrix:

[[ 46   0   0   0   0   0   0]
 [  0  35   0   0   0   0   0]
 [  0   0  32   0   0   0   0]
 [  3   0   0  84   1   0   1]
 [  0   0   0   0 334   0   3]
 [  0   0   0   0   1   8   0]
 [  0   0   0   0   2   0 642]]

ANN Model Saved Successfully

Enter CSV File Name: Dummy Test Data.csv
Total Rows in Uploa

In [ ]:
# =========================================================
# METER REMARK CLASSIFICATION USING ANN
# =========================================================

# =========================
# 1 IMPORT LIBRARIES
# =========================

import pandas as pd
import pickle
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical


# =========================
# 2 LOAD DATASET
# =========================

data = pd.read_csv("dataset.csv")

print("Dataset Loaded Successfully")

print(data.head())


# =========================
# 3 INPUT AND OUTPUT
# =========================

X = data['TEXT_REMARKS']

y = data['ACTION_CATEGORY']


# =========================
# 4 TF-IDF VECTORIZATION
# =========================

tfidf = TfidfVectorizer(
    stop_words='english',
    lowercase=True
)

X_tfidf = tfidf.fit_transform(X)

print("\nTF-IDF Completed")

print("Total Features:", X_tfidf.shape[1])


# =========================
# 5 LABEL ENCODING
# =========================

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

# Convert into categorical
y_categorical = to_categorical(y_encoded)

print("\nLabel Encoding Completed")


# =========================
# 6 TRAIN TEST SPLIT
# =========================

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y_categorical,
    test_size=0.2,
    random_state=42
)

print("\nDataset Split Completed")


# =========================
# 7 BUILD ANN MODEL
# =========================

model = Sequential()

# Input Layer
model.add(Dense(
    128,
    activation='relu',
    input_shape=(X_train.shape[1],)
))

# Hidden Layer
model.add(Dense(
    64,
    activation='relu'
))

# Output Layer
model.add(Dense(
    y_categorical.shape[1],
    activation='softmax'
))

print("\nANN Model Created")


# =========================
# 8 COMPILE MODEL
# =========================

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("\nModel Compilation Completed")


# =========================
# 9 TRAIN MODEL
# =========================

history = model.fit(
    X_train.toarray(),
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2
)

print("\nANN Training Completed")


# =========================
# 10 MODEL PREDICTION
# =========================

y_pred_probs = model.predict(X_test.toarray())

y_pred = np.argmax(y_pred_probs, axis=1)

y_test_actual = np.argmax(y_test, axis=1)


# =========================
# 11 ACCURACY
# =========================

accuracy = accuracy_score(
    y_test_actual,
    y_pred
)

print("\n==============================")
print("ANN MODEL ACCURACY")
print("==============================")

print("Accuracy:", accuracy)


# =========================
# 12 CLASSIFICATION REPORT
# =========================

print("\nClassification Report:\n")

print(
    classification_report(
        y_test_actual,
        y_pred
    )
)


# =========================
# 13 CONFUSION MATRIX
# =========================

print("\nConfusion Matrix:\n")

print(
    confusion_matrix(
        y_test_actual,
        y_pred
    )
)


# =========================
# 14 SAVE MODEL
# =========================

model.save("ann_meter_model.h5")

pickle.dump(tfidf, open("ann_tfidf.pkl", "wb"))

pickle.dump(label_encoder, open("label_encoder.pkl", "wb"))

print("\nANN Model Saved Successfully")


# =========================================================
# 15 LIVE PREDICTION FUNCTION
# =========================================================

def predict_department(text):

    # TF-IDF Transform
    text_vector = tfidf.transform([str(text)])

    # Predict Probability
    prediction_prob = model.predict(
        text_vector.toarray()
    )

    # Get Highest Probability Index
    predicted_index = np.argmax(prediction_prob)

    # Decode Department Name
    predicted_department = label_encoder.inverse_transform(
        [predicted_index]
    )[0]

    # Confidence Score
    confidence = np.max(prediction_prob) * 100

    return predicted_department, confidence


# =========================================================
# 16 BULK CSV PREDICTION
# =========================================================

input_file = input("\nEnter CSV File Name: ")

bulk_data = pd.read_csv(input_file)

print("Total Rows in Uploaded CSV:", len(bulk_data))

predicted_departments = []

confidence_scores = []

confidence_ranges = []

match_status = []


# =========================================================
# PREDICTION LOOP
# =========================================================

for index, row in bulk_data.iterrows():

    remark = row['TEXT_REMARKS']

    department, confidence = predict_department(remark)

    confidence = round(confidence, 2)

    predicted_departments.append(department)

    confidence_scores.append(confidence)

    # =========================================
    # CONFIDENCE RANGE
    # =========================================

    if confidence >= 90:

        confidence_ranges.append("100-90% Very High")

    elif confidence >= 75:

        confidence_ranges.append("90-75% High")

    elif confidence >= 50:

        confidence_ranges.append("75-50% Medium")

    else:

        confidence_ranges.append("Below 50% Low")

    # =========================================
    # MATCH STATUS
    # =========================================

    actual = str(row['ACTION_CATEGORY']).strip()

    predicted = str(department).strip()

    if actual == predicted:

        match_status.append("Correct")

    else:

        match_status.append("Wrong")


# =========================================================
# ADD OUTPUT COLUMNS
# =========================================================

bulk_data['PREDICTED_DEPARTMENT'] = predicted_departments

bulk_data['CONFIDENCE_SCORE'] = confidence_scores

bulk_data['CONFIDENCE_RANGE'] = confidence_ranges

bulk_data['MATCH_STATUS'] = match_status

bulk_data['IS_MATCH'] = (
    bulk_data['ACTION_CATEGORY']
    ==
    bulk_data['PREDICTED_DEPARTMENT']
)


# =========================================================
# SAVE OUTPUT CSV
# =========================================================

bulk_data.to_csv(
    "ANN_Predicted_Output.csv",
    index=False
)

print("\n====================================")
print("ANN PREDICTION COMPLETED")
print("====================================")

print("\nOutput File Saved Successfully")

Dataset Loaded Successfully
                                        TEXT_REMARKS  \
0  Consumer want Some time for meter testing as h...   
1  Consumer going to Village Consumer want some t...   
2  as per telephonic talk with consumer consumer ...   
3  No person at side and no response on said phon...   
4  No person at side and no response on said phon...   

                           ACTION_CATEGORY  Unnamed: 2  Unnamed: 3  \
0  Back-Office / Administrative Operations         NaN         NaN   
1  Back-Office / Administrative Operations         NaN         NaN   
2  Back-Office / Administrative Operations         NaN         NaN   
3  Back-Office / Administrative Operations         NaN         NaN   
4  Back-Office / Administrative Operations         NaN         NaN   

   Unnamed: 4  Unnamed: 5  Unnamed: 6  Unnamed: 7  Unnamed: 8  Unnamed: 9  \
0         NaN         NaN         NaN         NaN         NaN         NaN   
1         NaN         NaN         NaN         NaN         Na

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



ANN Model Created

Model Compilation Completed
Epoch 1/10
120/120 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - accuracy: 0.8339 - loss: 0.7043 - val_accuracy: 0.9801 - val_loss: 0.1432
Epoch 2/10
120/120 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9801 - loss: 0.0932 - val_accuracy: 0.9916 - val_loss: 0.0514
Epoch 3/10
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9942 - loss: 0.0413 - val_accuracy: 0.9895 - val_loss: 0.0400
Epoch 4/10
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9961 - loss: 0.0255 - val_accuracy: 0.9906 - val_loss: 0.0361
Epoch 5/10
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9963 - loss: 0.0206 - val_accuracy: 0.9895 - val_loss: 0.0321
Epoch 6/10
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9971 - loss: 0.0172 - val_accuracy: 0.9895 - val_loss: 0.0329
Epoch 7/10
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9974 - loss: 0.0156 - val_accuracy: 0.9916 - val_loss: 0.0320
Epoch 8/10
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - a


ANN MODEL ACCURACY
Accuracy: 0.9907718120805369

Classification Report:

              precision    recall  f1-score   support

           0       0.94      1.00      0.97        46
           1       1.00      1.00      1.00        35
           2       1.00      1.00      1.00        32
           3       1.00      0.94      0.97        89
           4       0.99      0.99      0.99       337
           5       1.00      0.89      0.94         9
           6       0.99      1.00      1.00       644

    accuracy                           0.99      1192
   macro avg       0.99      0.97      0.98      1192
weighted avg       0.99      0.99      0.99      1192


Confusion Matrix:

[[ 46   0   0   0   0   0   0]
 [  0  35   0   0   0   0   0]
 [  0   0  32   0   0   0   0]
 [  3   0   0  84   1   0   1]
 [  0   0   0   0 334   0   3]
 [  0   0   0   0   1   8   0]
 [  0   0   0   0   2   0 642]]

ANN Model Saved Successfully

Enter CSV File Name: Dummy Test Data.csv
Total Rows in Uploa

# **Link for Model**

In [8]:
# Link: https://colab.research.google.com/drive/1SSWYgDCG0-kTyS4v1whcSXzqxCQalfCb?usp=sharing